# Installing AutoGluon

This section installs the AutoGluon library, which is a powerful AutoML framework designed to automate machine learning tasks. The `pip install autogluon` command downloads and installs the AutoGluon package along with its dependencies. This step is crucial to enable the use of AutoGluon's tabular prediction capabilities for our backpack price prediction task.

In [ ]:
pip install autogluon

# Importing Libraries

Here, we import the necessary Python libraries for our task:
- `TabularDataset` and `TabularPredictor` from `autogluon.tabular` are used for handling tabular data and building the predictive model.
- `pandas` is used for data manipulation and loading CSV files.
- `train_test_split` from `sklearn.model_selection` is used to split the dataset into training and validation sets.

In [ ]:
# Import necessary modules
from autogluon.tabular import TabularDataset, TabularPredictor
import pandas as pd
from sklearn.model_selection import train_test_split

# Loading the Data

This section loads the datasets required for training and testing:
- `train.csv`: The primary training dataset containing features and the target variable (Price).
- `training_extra.csv`: An additional training dataset to augment the training data.
- `test.csv`: The test dataset for which we need to predict the Price.

The datasets are loaded using `pandas.read_csv`, with the `id` column set as the index for each DataFrame.

In [ ]:
# Load and prepare the data
train = pd.read_csv("/kaggle/input/playground-series-s5e2/train.csv", index_col='id')
train_extra = pd.read_csv("/kaggle/input/playground-series-s5e2/training_extra.csv", index_col='id')
test = pd.read_csv('/kaggle/input/playground-series-s5e2/test.csv', index_col='id')

# Defining the Feature Engineering Function

This cell defines a function `add_features` to perform feature engineering on the datasets:
- Creates a new feature `Weight_per_Compartment` by dividing `Weight Capacity (kg)` by `Compartments` to capture the weight capacity per compartment.
- Converts binary categorical features (`Laptop Compartment` and `Waterproof`) from 'Yes'/'No' to numeric values (1/0) for model compatibility.

This function enhances the dataset by adding meaningful features that may improve model performance.

In [ ]:
# Feature engineering function
def add_features(df):
    # Create interaction features
    df['Weight_per_Compartment'] = df['Weight Capacity (kg)'] / df['Compartments']
    
    # Convert binary features to numeric
    binary_map = {'Yes': 1, 'No': 0}
    df['Laptop Compartment'] = df['Laptop Compartment'].map(binary_map)
    df['Waterproof'] = df['Waterproof'].map(binary_map)
    
    return df

# Applying Feature Engineering

The `add_features` function is applied to the `train`, `train_extra`, and `test` datasets to ensure consistency in feature engineering across all datasets. This step adds the `Weight_per_Compartment` feature and converts binary features to numeric values in all datasets.

In [ ]:
# Apply feature engineering to all datasets
train = add_features(train)
train_extra = add_features(train_extra)
test = add_features(test)

# Combining Training Datasets

This cell combines the `train` and `train_extra` datasets into a single DataFrame `df` using `pd.concat`. The `axis=0` parameter stacks the datasets vertically, and `ignore_index=True` creates a new index for the combined dataset. This step increases the amount of training data, potentially improving model performance.

In [ ]:
# Combine train and train extra data sets
df = pd.concat([train, train_extra], axis=0, ignore_index=True)

# Defining the Target Column

The target column for the prediction task is defined as `Price`. This variable will be used as the label that the model will predict.

In [ ]:
# Define target column
target = 'Price'

# Splitting Data into Training and Validation Sets

The combined dataset `df` is split into training (`train_data`) and validation (`val_data`) sets using `train_test_split`. A 20% validation split is used (`test_size=0.2`), and `random_state=42` ensures reproducibility. This split allows the model to be trained on one portion of the data and evaluated on a separate portion to assess its performance.

In [ ]:
# Split data into train and validation sets
train_data, val_data = train_test_split(df, test_size=0.2, random_state=42)

# Training the AutoGluon Model

This cell initializes and trains an AutoGluon `TabularPredictor` for regression:
- **Parameters**:
  - `label=target`: Specifies `Price` as the target variable.
  - `problem_type='regression'`: Indicates this is a regression task.
  - `eval_metric='root_mean_squared_error'`: Uses RMSE to evaluate model performance.
  - `path='ag_models_backpack'`: Saves the trained models to this directory.
- **Training Configuration**:
  - `presets='medium_quality'`: Uses a medium-quality preset for faster training compared to the default best-quality preset.
  - `time_limit=600`: Limits training to 10 minutes (600 seconds) to manage computational resources.
  - `hyperparameters='default'`: Uses default hyperparameters without tuning to save time.
  - `excluded_model_types=['KNN', 'NN_TORCH', 'FASTAI']`: Excludes slower models (KNN, neural networks, and FastAI) to speed up training.
  - `verbosity=2`: Provides standard logging output for monitoring.

The model is trained on `train_data` and validated on `val_data`. AutoGluon automatically tries multiple models and creates an ensemble for optimal performance.

In [ ]:
# Initialize AutoGluon predictor with time constraints
predictor = TabularPredictor(
    label=target,
    problem_type='regression',
    eval_metric='root_mean_squared_error',
    path='ag_models_backpack'
).fit(
    train_data=train_data,
    tuning_data=val_data,
    # Use medium_quality preset instead of best_quality for faster training
    presets='medium_quality',
    # Set a strict 10-minute time limit (600 seconds)
    time_limit=600,
    # Skip hyperparameter tuning to save time
    hyperparameters='default',
    # Limit model types to faster ones
    excluded_model_types=['KNN', 'NN_TORCH', 'FASTAI'],
    verbosity=2
)

# Evaluating Model Performance

The trained model is evaluated on the validation set (`val_data`) using the `predictor.evaluate` method. This returns a dictionary of performance metrics, including the root mean squared error (RMSE), mean squared error (MSE), mean absolute error (MAE), R², Pearson correlation, and median absolute error. The results are printed to assess how well the model generalizes to unseen data.

In [ ]:
# Evaluate on validation data
performance = predictor.evaluate(val_data)
print("Validation performance:", performance)

# Generating Predictions on Test Data

The trained model is used to predict the `Price` for the test dataset using the `predictor.predict` method. The predictions are stored in `test_pred` for further processing.

In [ ]:
# Generate predictions on test data
test_pred = predictor.predict(test)

# Creating the Submission File

This cell creates a submission file by combining the test dataset's `id` column with the predicted `Price` values. The resulting DataFrame is saved as `submission.csv` with the `id` column set as the index. A confirmation message is printed to indicate successful creation of the submission file.

In [ ]:
# Create submission file
submission = pd.DataFrame({'id': test.index, 'Price': test_pred})
submission.set_index('id', inplace=True)
submission.to_csv('submission.csv')
print("Submission file created")